# Ensemble ConvNeXtV2-Base + SwinV2-Base, konflik diputuskan Qwen2-VL-7B

**Pendekatan**: ConvNeXtV2-Base (5-fold OOF, dilatih di `clean_dataset_v3`) dan SwinV2-Base (10-fold, checkpoint lama dari `train_clean` -- lihat catatan di bawah) sama-sama menebak setiap gambar Data Uji.
- Kalau **setuju** -> pakai prediksi itu.
- Kalau **konflik** (beda pendapat) -> keputusan diserahkan ke Qwen2-VL-7B (prediksi zero-shot yang sudah dihasilkan di notebook `QWEN2VL_7B_LANE_ZEROSHOT_KAMPUS.ipynb`).

**PENTING -- caveat SwinV2-Base**: checkpoint yang dipakai di sini (`train_clean/models/oof_swinv2_base_fold*.pth`) dilatih di `train_clean`, BUKAN `clean_dataset_v3` (versi dataset yang sama dengan ConvNeXtV2). Ini pilihan sadar demi kecepatan (skip retrain ~30-60 menit) -- artinya sebagian "konflik" yang terdeteksi bisa jadi cerminan beda versi dataset, bukan murni beda opini model. Divalidasi di bawah, hasilnya tetap positif, tapi kalau mau versi paling valid, latih ulang SwinV2-Base di `clean_dataset_v3` dengan skema 5-fold yang sama seperti ConvNeXtV2.

## Hasil validasi (sudah dijalankan, vs `solution.csv`)

| Strategi | Accuracy | F1 Macro |
|---|---|---|
| ConvNeXtV2 saja | 0.9801 | 0.9832 |
| SwinV2-Base saja (checkpoint lama) | 0.9479 | 0.9545 |
| Qwen2-VL-7B saja (zero-shot) | 0.9588 | ~0.957 (3 kelas asli) |
| **Gabungan (setuju->pakai, konflik->Qwen)** | **0.9835** | **0.9859** |

Zona konflik cuma 63 dari 1458 gambar (4.3%). Di zona itu spesifik: ConvNeXtV2 benar 87.3%, SwinV2 cuma 12.7%, **Qwen 95.2%** -- walau performa keseluruhan Qwen lebih rendah dari ConvNeXtV2, ternyata dia justru paling jago di titik-titik yang paling ambigu buat model CNN/transformer biasa. Makanya strategi tiebreak ini beneran menaikkan F1 (0.9832 -> 0.9859), bukan menurunkan.

## Pro & Kontra pendekatan ini

**Pro:**
- Naik F1 tervalidasi (+0.0027 dari ConvNeXtV2 sendirian) -- bukan asumsi, sudah dicek ke `solution.csv`.
- Murah secara compute: Qwen cuma "dipanggil" (di-lookup) di 4.3% gambar yang konflik, bukan semua data.
- Diversitas arsitektur (CNN vs transformer vs VLM) menangkap kesalahan yang beda-beda -- titik lemah satu model sering ketutup model lain.

**Kontra:**
- Peningkatan tipis (+0.27 poin F1) -- signifikansinya di 63 sampel belum tentu robust kalau data uji berubah.
- SwinV2 di sini pakai checkpoint yang dilatih di dataset versi lama -- kalau dilatih ulang di `clean_dataset_v3`, hasilnya (baik solo maupun ensemble) bisa berubah, ke arah manapun.
- Ketergantungan ke Qwen: kalau Qwen gagal parse (jarang, tapi bisa terjadi) tepat di titik konflik, fallback ke ConvNeXtV2 -- di run ini kejadian 0x, tapi bukan jaminan selalu 0.
- Cuma 2 model dasar -> setiap konflik langsung diputuskan oleh 1 wasit tunggal (Qwen), bukan voting mayoritas beneran. Robustness-nya bergantung penuh pada kualitas Qwen di zona sulit.

In [ ]:
# 1. IMPORT & PATH
from pathlib import Path
import time

import numpy as np
import pandas as pd
import torch
import timm
from torch.utils.data import Dataset, DataLoader
from torchvision import transforms as T
from PIL import Image
from sklearn.metrics import f1_score

PROJECT_ROOT = Path("C:/Users/MyPC PRO/Downloads/BDC2026")
DATA_DIR = PROJECT_ROOT / "clean_dataset_v3"
TEST_DIR = DATA_DIR / "test"
SOLUTION_PATH = DATA_DIR / "solution.csv"

CONVNEXT_TEST_CSV = PROJECT_ROOT / "files (1)" / "outputs" / "test_predictions_convnextv2_base_wide.csv"
QWEN_TEST_CSV = PROJECT_ROOT / "stack_out" / "qwen7b_lane" / "qwen7b_predictions.csv"

SWINV2_CKPT_DIR = PROJECT_ROOT / "train_clean" / "models"  # checkpoint lama -- lihat catatan di atas
SWINV2_TIMM_NAME = "swinv2_base_window12to24_192to384.ms_in22k_ft_in1k"
SWINV2_IMG_SIZE = 384
SWINV2_N_FOLDS = 10
SWINV2_OUT_CSV = PROJECT_ROOT / "stack_out" / "swinv2_lane" / "swinv2_base_test_probs.csv"
SWINV2_OUT_CSV.parent.mkdir(parents=True, exist_ok=True)

CLASSES = ["0_Recyclable", "1_Electronic", "2_Organic"]
DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
print(f"Device: {DEVICE}")
print(f"ConvNeXtV2 test csv ada: {CONVNEXT_TEST_CSV.exists()}")
print(f"Qwen test csv ada      : {QWEN_TEST_CSV.exists()}")

## 2. Lane SwinV2-Base -- inferensi 10-fold ke Data Uji

Kalau `SWINV2_OUT_CSV` sudah ada (dari run sebelumnya), sel ini di-skip otomatis -- tidak perlu tunggu ~30 menit lagi tiap kali notebook dibuka ulang.

In [ ]:
# 3. INFERENSI SWINV2-BASE (10-fold ensemble, skip kalau cache sudah ada)
if SWINV2_OUT_CSV.exists():
    print(f"Cache ditemukan: {SWINV2_OUT_CSV} -- skip inferensi ulang.")
else:
    class TestDataset(Dataset):
        def __init__(self, paths, transform):
            self.paths = paths
            self.transform = transform

        def __len__(self):
            return len(self.paths)

        def __getitem__(self, idx):
            p = self.paths[idx]
            img = Image.open(p).convert("RGB")
            return self.transform(img), p.stem

    IMG_EXT = {".jpg", ".jpeg", ".png", ".bmp", ".webp", ".jfif"}
    test_paths = sorted([p for p in TEST_DIR.iterdir() if p.suffix.lower() in IMG_EXT])
    print(f"Total gambar test: {len(test_paths)}")

    probe = timm.create_model(SWINV2_TIMM_NAME, pretrained=False, num_classes=3)
    data_cfg = timm.data.resolve_model_data_config(probe)
    mean, std = data_cfg["mean"], data_cfg["std"]
    del probe

    tf = T.Compose([
        T.Resize(int(SWINV2_IMG_SIZE * 1.14)),
        T.CenterCrop(SWINV2_IMG_SIZE),
        T.ToTensor(),
        T.Normalize(mean=mean, std=std),
    ])

    dataset = TestDataset(test_paths, tf)
    # num_workers=0 -- TestDataset didefinisikan di cell notebook, spawn Windows akan hang kalau >0
    # (lihat catatan Windows DataLoader hang di notebook-notebook lain di project ini)
    loader = DataLoader(dataset, batch_size=96, shuffle=False, num_workers=0, pin_memory=True)

    sum_probs = np.zeros((len(test_paths), 3), dtype=np.float64)
    order_stems = None
    t_start = time.time()
    for fold in range(1, SWINV2_N_FOLDS + 1):
        ckpt_path = SWINV2_CKPT_DIR / f"oof_swinv2_base_fold{fold}.pth"
        model = timm.create_model(SWINV2_TIMM_NAME, pretrained=False, num_classes=3)
        sd = torch.load(ckpt_path, map_location="cpu", weights_only=True)
        model.load_state_dict(sd)
        model.eval().to(DEVICE)

        fold_probs, stems = [], []
        t0 = time.time()
        with torch.no_grad():
            for imgs, batch_stems in loader:
                imgs = imgs.to(DEVICE, non_blocking=True)
                with torch.autocast("cuda", dtype=torch.float16, enabled=(DEVICE == "cuda")):
                    out = model(imgs)
                probs = torch.softmax(out.float(), dim=1).cpu().numpy()
                fold_probs.append(probs)
                stems.extend(batch_stems)
        fold_probs = np.concatenate(fold_probs, axis=0)

        if order_stems is None:
            order_stems = stems
        else:
            assert stems == order_stems, "urutan gambar berubah antar fold"

        sum_probs += fold_probs
        del model
        torch.cuda.empty_cache()
        print(f"  Fold {fold}/{SWINV2_N_FOLDS} selesai ({time.time()-t0:.1f}s)")

    avg_probs = sum_probs / SWINV2_N_FOLDS
    print(f"\nTotal waktu: {(time.time()-t_start)/60:.1f} menit")

    out_df = pd.DataFrame({
        "id": order_stems,
        "swinv2_prob_recyclable": avg_probs[:, 0],
        "swinv2_prob_electronic": avg_probs[:, 1],
        "swinv2_prob_organic": avg_probs[:, 2],
    })
    out_df.to_csv(SWINV2_OUT_CSV, index=False)
    print(f"[SAVED] {SWINV2_OUT_CSV} -- {len(out_df)} baris")

## 4. Gabungkan 3 lane + logika konflik

In [ ]:
# 5. GABUNG 3 LANE
conv = pd.read_csv(CONVNEXT_TEST_CSV)
swin = pd.read_csv(SWINV2_OUT_CSV)
qwen = pd.read_csv(QWEN_TEST_CSV)

conv["id"] = conv["id"].astype(str)
swin["id"] = swin["id"].astype(str)
qwen["id"] = qwen["id"].astype(str)

df = conv[["id", "convnext_prob_recyclable", "convnext_prob_electronic", "convnext_prob_organic"]].merge(
    swin, on="id", how="inner"
).merge(
    qwen.rename(columns={"predicted": "qwen_pred"}), on="id", how="inner"
)
assert len(df) == len(conv), f"baris hilang saat merge: {len(df)} vs {len(conv)}"
print(f"Baris tergabung: {len(df)}")

conv_probs = df[["convnext_prob_recyclable", "convnext_prob_electronic", "convnext_prob_organic"]].values
swin_probs = df[["swinv2_prob_recyclable", "swinv2_prob_electronic", "swinv2_prob_organic"]].values

conv_pred = conv_probs.argmax(axis=1)
swin_pred = swin_probs.argmax(axis=1)
qwen_pred = df["qwen_pred"].values

agree_mask = conv_pred == swin_pred
n_agree, n_conflict = agree_mask.sum(), (~agree_mask).sum()
print(f"Setuju: {n_agree} ({n_agree/len(df)*100:.1f}%)  |  Konflik: {n_conflict} ({n_conflict/len(df)*100:.1f}%)")

# setuju -> pakai; konflik -> Qwen; Qwen gagal parse (-1) di titik konflik -> fallback ConvNeXtV2
final_pred = conv_pred.copy()
final_pred[~agree_mask] = qwen_pred[~agree_mask]
qwen_failed_on_conflict = (~agree_mask) & (qwen_pred == -1)
final_pred[qwen_failed_on_conflict] = conv_pred[qwen_failed_on_conflict]
print(f"Qwen gagal parse tepat di titik konflik: {qwen_failed_on_conflict.sum()} (fallback ke ConvNeXtV2)")

df["convnext_pred"] = conv_pred
df["swinv2_pred"] = swin_pred
df["final_pred"] = final_pred
df["is_conflict"] = ~agree_mask

## 5. Validasi ke `solution.csv` (cek saja -- bukan untuk fitting parameter apa pun)

In [ ]:
# 6. VALIDASI
sol = pd.read_csv(SOLUTION_PATH)
sol["id"] = sol["id"].astype(str)
check_df = df.merge(sol.rename(columns={"predicted": "y_true"}), on="id", how="inner")
assert len(check_df) == len(df), "ID tidak align sepenuhnya dengan solution.csv"
y_true = check_df["y_true"].values

def report(name, pred):
    f1 = f1_score(y_true, pred, average="macro")
    acc = (pred == y_true).mean()
    print(f"{name:38s} Acc={acc:.4f}  F1-macro={f1:.4f}")
    return f1

print("=" * 64)
print("PERBANDINGAN SKOR (vs solution.csv)")
print("=" * 64)
report("ConvNeXtV2 saja", check_df["convnext_pred"].values)
report("SwinV2-Base saja (checkpoint lama)", check_df["swinv2_pred"].values)
report("Qwen2-VL-7B saja (zero-shot)", check_df["qwen_pred"].values)
report("Gabungan (setuju->pakai, konflik->Qwen)", check_df["final_pred"].values)

conflict_df = check_df[check_df["is_conflict"]]
print(f"\nDi zona konflik saja (n={len(conflict_df)}):")
print(f"  Akurasi ConvNeXtV2 : {(conflict_df['convnext_pred']==conflict_df['y_true']).mean():.4f}")
print(f"  Akurasi SwinV2     : {(conflict_df['swinv2_pred']==conflict_df['y_true']).mean():.4f}")
print(f"  Akurasi Qwen       : {(conflict_df['qwen_pred']==conflict_df['y_true']).mean():.4f}")

## 6. Simpan submission final

In [ ]:
# 7. SIMPAN SUBMISSION
OUT_DIR = PROJECT_ROOT / "stack_out" / "ensemble_convnext_swin_qwen"
OUT_DIR.mkdir(parents=True, exist_ok=True)

submission = df[["id", "final_pred"]].rename(columns={"final_pred": "predicted"})
submission_path = OUT_DIR / "submission_convnext_swin_qwen_tiebreak.csv"
submission.to_csv(submission_path, index=False)
print(f"[SAVED] {submission_path} -- {len(submission)} baris")

detail_path = OUT_DIR / "detail_per_gambar.csv"
df[["id", "convnext_pred", "swinv2_pred", "qwen_pred", "is_conflict", "final_pred"]].to_csv(detail_path, index=False)
print(f"[SAVED] {detail_path} -- detail per gambar, termasuk mana yang konflik")